In [2]:
import numpy as np


def MembraneSolution(nodcoor, eletyp, elenod, elemat, elefab, eleopt, doftag, dofval):
    """
    INPUT: nodcoor (list of coords [[x1,y1], [x2, y2], etc.])
           eletyp (list of element type [[ etyp ], [ etyp ], etc])
           elenod (connectivity, list of nodes for each element)
           elemat (list with E, ν , for each element [[E, ν], etc]
           elefab (list wiht plate thickness per element)
           eleopt (kept for faithfulness to Felippa, specifies float number in Mathematica)
           doftag (freedom tags, like nodcoor but 0 means force is specified, else 1. [[0,0], [1,0], etc])
           dofval (freedom values, lists either force or prescribed displacements, depends on doftag)

    OUTPUT: u (displacement vector)
            f (force vector)
            sig (stresses)
    """
    numnod = len(nodcoor)
    numele = len(elenod)

    K = MembraneMasterStiffness(nodcoor, eletyp, elenod, elemat, elefab, eleo)
    ns = 0
    for n in range(numnod):
        for j in range(2):
            if doftag[n][j] > 0:
                ns += 1

    supdof = np.zeros(ns, dtype=int)  
    supval = np.zeros(ns)              

    k = 0
    for n in range(numnod):
        for j in range(2):
            if doftag[n][j] > 0:
                supdof[k] = 2 * n + (j + 1)   
                supval[k] = dofval[n][j]
                k += 1

    f = ModifiedNodeForces(supdof, supval, K, np.asarray(dofval).flatten())
    Kmod = ModifiedMasterStiffness(supdof, K)

    u = np.linalg.solve(Kmod, f)
    u = np.where(np.abs(u) < 1e-10, 0, u)

    f = K @ u
    f = np.where(np.abs(f) < 1e-10, 0, f)

    sig = MembraneNodalStresses(nodcoor, eletyp, elenod, elemat, elefab, eleopt, u)
    sig = np.where(np.abs(sig) < 1e-10, 0, sig)

    return u, f, sig


def MembraneMasterStiffness(nodcoor, eletyp, elenod, elemat, elefab, eleopt):
    """
    INPUT: nodcoor (list of coords [[x1,y1], [x2, y2], etc.])
           eletyp (list of element type [[ etyp ], [ etyp ], etc])
           elenod (connectivity, list of nodes for each element)
           elemat (list with E, ν , for each element [[E, ν], etc]
           elefab (list wiht plate thickness per element)
           eleopt (kept for faithfulness to Felippa, specifies float number in Mathematica)

    OUTPUT: K (master stiffness matrix)
    """
    numele = len(elenod)
    numnod = len(nodcoor)

    nodcoor = np.asarray(nodcoor, dtype=float)
    assert nodcoor.ndim == 2 and nodcoor.shape[1] == 2, \
        f"nodcoor must be (numnod, 2), got {nodcoor.shape}"
    
    K = np.zeros((2 * numnod, 2 * numnod))
    numer = eleopt[0]

    for ne in range(numele):                       
        type_specific = eletyp[ne]                 # in original, variable is named "type"

        if type_specific not in ["Trig3", "Trig6", "Trig10", "Quad4", "Quad9"]:
            print(f"Illegal element type, ne={ne}")
            return None

        eNL = elenod[ne]
        n = len(eNL)

        eftab = []
        for e in eNL:
            eftab += [2 * e - 1, 2 * e]

        ncoor = []
        for e in eNL:
            ncoor.append(nodcoor[e - 1])
            
        Em, nu = elemat[ne]                         
        Emat = Em / (1 - nu ** 2) * np.array([
            [1,  nu, 0],
            [nu, 1,  0],
            [0,  0,  (1 - nu) / 2],
        ])

        th = elefab[ne][0]  
        
        if type_specific == "Trig3":
            Ke = Trig3IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer])
        elif type_specific == "Quad4":
            Ke = Quad4IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer, 2])
        elif type_specific == "Trig6":
            Ke = Trig6IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer, 3])
        elif type_specific == "Quad9":
            Ke = Quad9IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer, 3])
        elif type_specific == "Trig10":
            Ke = Trig10IsoPMembraneStiffness(ncoor, [Emat, 0, 0], [th], [numer, 3])

        neldof = len(Ke)

        # assemble upper triangle, then mirror it into lower 
        for i in range(neldof):
            ii = eftab[i]
            for j in range(i, neldof):
                jj = eftab[j]
                K[ii - 1, jj - 1] += Ke[i, j]
                K[jj - 1, ii - 1] = K[ii - 1, jj - 1]

    return K



import numpy as np


def MembraneNodalStresses(nodcoor, eletyp, elenod, elemat, elefab, eleopt, u):
    """
    INPUT: nodcoor (list of coords [[x1,y1], [x2, y2], etc.])
           eletyp (list of element type [[ etyp ], [ etyp ], etc])
           elenod (connectivity, list of nodes for each element)
           elemat (list with E, ν , for each element [[E, ν], etc]
           elefab (list wiht plate thickness per element)
           eleopt (kept for faithfulness to Felippa, specifies float number in Mathematica)
           u (displacement vector)

    OUTPUT: sig (nodal stresses, averaged over the elements at each node,
                 one [sigxx, sigyy, sigxy] row per node)
    """
    numele = len(elenod)
    numnod = len(nodcoor)

    ncount = np.zeros(numnod, dtype=int)
    numer = eleopt[0]

    sige = [0] * numele
    sig = np.zeros((numnod, 3))

    for ne in range(numele):
        type_specific = eletyp[ne]              # in original, variable is named "type"

        eNL = elenod[ne]
        n = len(eNL)
        ue = np.zeros(2 * n)

        eftab = []
        for e in eNL:
            eftab += [2 * e - 1, 2 * e]

        ncoor = []
        for e in eNL:
            ncoor.append(nodcoor[e - 1])

        for i in range(2 * n):
            ii = eftab[i]
            ue[i] = u[ii - 1]                    # python index

        Em, nu = elemat[ne]
        Emat = Em / (1 - nu ** 2) * np.array([
            [1,  nu, 0],
            [nu, 1,  0],
            [0,  0,  (1 - nu) / 2],
        ])

        th = elefab[ne][0]

        if type_specific == "Trig3":
            esig = Trig3IsoPMembraneStresses(ncoor, [Emat, 0, 0], [th], [numer], ue)
        elif type_specific == "Quad4":
            esig = Quad4IsoPMembraneStresses(ncoor, [Emat, 0, 0], [th], [numer, 2], ue)
        elif type_specific == "Trig6":
            esig = Trig6IsoPMembraneStresses(ncoor, [Emat, 0, 0], [th], [numer, 3], ue)
        elif type_specific == "Quad9":
            esig = Quad9IsoPMembraneStresses(ncoor, [Emat, 0, 0], [th], [numer, 3], ue)
        elif type_specific == "Trig10":
            esig = Trig10IsoPMembraneStresses(ncoor, [Emat, 0, 0], [th], [numer, 3], ue)

        esig = np.where(np.abs(esig) < 1e-10, 0, esig)   # Chop
        sige[ne] = esig

        for i in range(n):
            k = eNL[i]
            ncount[k - 1] += 1
            for j in range(3):
                sig[k - 1, j] += esig[i, j]

    for n in range(numnod):
        k = ncount[n]
        if k > 1:
            sig[n] = sig[n] / k

    return sig

In [5]:
# these functions were previously defined in chapter 23. Added here as they are called above.

def QuadGaussRuleInfo(rule, point):
    """
    INPUT: pth rule, point 
    PROCESS: calculates Legendre-Gauss in 2x2 
    OUTPUT: (xi, eta), weight (multiplied)
    """
    # allows for specifying amount of points per axis
        
    if np.size(rule) == 2:
        p1, p2 = rule
    else:
        p1 = p2 = rule
    
    if np.size(point) == 2:
        i1, i2 = point
    else:
        k = point
    
        # visiting counter
        i2 = (k - 1) // p1 + 1
        i1 = k - p1 * (i2 - 1)
            
    xi, w1 = LineGaussRuleInfo(p1, i1)
    eta, w2 = LineGaussRuleInfo(p2, i2)
    
    info = np.array([xi, eta], dtype=float), w1 * w2

    return info

def LineGaussRuleInfo(rule, point):
    """
    INPUT: pth rule, ith point 
    PROCESS: calculates the ith point of the Legendre-Gauss quadrature for a pth rule (up to the 4th rule)
    OUTPUT: point, weight as "info"
    """

    g2 = np.array([-1, 1]) / np.sqrt(3)

    w3 = np.array([5/9, 8/9, 5/9])
    g3 = np.array([-np.sqrt(3/5), 0, np.sqrt(3/5)])

    w4 = np.array([
        1/2 - np.sqrt(5/6)/6,
        1/2 + np.sqrt(5/6)/6,
        1/2 + np.sqrt(5/6)/6,
        1/2 - np.sqrt(5/6)/6
    ])

    g4 = np.array([
        -np.sqrt((3 + 2*np.sqrt(6/5))/7),
        -np.sqrt((3 - 2*np.sqrt(6/5))/7),
         np.sqrt((3 - 2*np.sqrt(6/5))/7),
         np.sqrt((3 + 2*np.sqrt(6/5))/7)
    ])

    # python index
    i = point - 1

    if rule == 1:
        info = np.array([0.0, 2.0])

    elif rule == 2:
        info = np.array([g2[i], 1.0])

    elif rule == 3:
        info = np.array([g3[i], w3[i]])

    elif rule == 4:
        info = np.array([g4[i], w4[i]])
    return info




def Quad4IsoPShapeFunDer(ncoor, qcoor):
    """
    INPUT: ncoor (node coordinates), qcoor (parametric coordinates)
    PROCESS:
    OUTPUT: Nf (shape functions), dNx, dNy (derivatives of the shape functions), Jdet (determinant of the jacobian)
    """
    xi, eta = qcoor
    
    Nf = 0.25 * np.array([
        (1-xi)*(1-eta),
        (1+xi)*(1-eta),
        (1+xi)*(1+eta),
        (1-xi)*(1+eta)
    ])

    dNxi = 0.25 * np.array([
        -(1-eta),
        (1-eta),
        (1+eta),
        -(1+eta)
    ])

    dNeta = 0.25 * np.array([
        -(1-xi),
        -(1+xi),
        (1+xi),
        (1-xi)
    ])

    ncoor = np.asarray(ncoor)

    x = ncoor[:, 0]
    y = ncoor[:, 1]

    J11 = np.dot(dNxi, x)
    J21 = np.dot(dNxi, y)
    J12 = np.dot(dNeta, x)
    J22 = np.dot(dNeta, y)

    Jdet = J11*J22 - J12*J21

    # to get ∂N/∂x, remember that chain rule gives ∂N/∂x= ∂N/∂ξ * ∂ξ/∂x + ∂N/∂η * ∂η/∂x, in matrix form, its thus the Jacobian inverse.

    dNx = ( J22 * dNxi - J21 * dNeta) / Jdet
    dNy = (-J12 * dNxi + J11 * dNeta) / Jdet

    return Nf,  dNx, dNy, Jdet




def Quad4IsoPMembraneStiffness(ncoor, mprop, fprop, options):
    """
    INPUT: ncoor (node coordinates), 
           mprop (material properties, given as list {Emat, rho, alpha}, note that Emat is 3x3 matrix), also rho and alpha are not used here)
           fprop (fabrication properties, i.e, thickness as four entry list, one entry list or empty list). If empty, it assumes uniform h = 1
           options (two item list, [numer, p], first isn't used here (its for symbolic op.), p specifies p points in each direction
    PROCESS:
    OUTPUT: Ke
    """
    Emat = np.asarray(mprop[0], dtype=float)

    if len(options) == 2:
        _, p = options
    else:
        p = 2

    th = 1.0
    
    if len(fprop) > 0:
        th = fprop[0]
    
    if p < 1 or p>4:
        print("p out of range")
        return None
    
    Ke = np.zeros((8, 8))

    for k in range(1, p*p + 1):
        
        qcoor, w = QuadGaussRuleInfo(p, k) # obtain isoparametric coordinates in the points where Gauss integration will be performed
        
        Nf, dNx, dNy, Jdet = Quad4IsoPShapeFunDer(ncoor, qcoor)


        if np.ndim(th) == 0: # constant thickness 
            h = th
        else:
            h = np.dot(th, Nf)

        c = w* Jdet * h

        #displacement strain matrix
        B = np.zeros((3,8))

        for i in range(4): # Felippa uses Flatten in Mathematica after creating tables with pairs (dNx, 0), which removes the braces. Here it's just a for loop
            B[0, 2*i]     = dNx[i]
            B[1, 2*i + 1] = dNy[i]
            B[2, 2*i]     = dNy[i]
            B[2, 2*i + 1] = dNx[i]

        Ke += c * (B.T @ Emat @ B)

    return Ke


In [7]:
s = [1, 0.70, 0.48, 0.30, 0.16, 0.07, 0.0]
 
xy1  = np.array([0, 6])
xy7  = np.array([0, 1])
xy8  = np.array([2.5, 6])
xy14 = np.array([np.cos(3 * np.pi / 8), np.sin(3 * np.pi / 8)])
xy21 = np.array([np.cos(np.pi / 4),     np.sin(np.pi / 4)])
xy15 = np.array([5, 6])
xy22 = np.array([5, 2])
xy28 = np.array([np.cos(np.pi / 8),     np.sin(np.pi / 8)])
xy29 = np.array([5, 0])
xy35 = np.array([1, 0])
 
NodeCoordinates = np.zeros((35, 2))
for n in range(1, 8):     # nodes 1..7
    NodeCoordinates[n - 1] = s[n - 1]  * xy1  + (1 - s[n - 1])  * xy7
for n in range(8, 15):    # nodes 8..14
    NodeCoordinates[n - 1] = s[n - 8]  * xy8  + (1 - s[n - 8])  * xy14
for n in range(15, 22):   # nodes 15..21
    NodeCoordinates[n - 1] = s[n - 15] * xy15 + (1 - s[n - 15]) * xy21
for n in range(22, 29):   # nodes 22..28
    NodeCoordinates[n - 1] = s[n - 22] * xy22 + (1 - s[n - 22]) * xy28
for n in range(29, 36):   # nodes 29..35
    NodeCoordinates[n - 1] = s[n - 29] * xy29 + (1 - s[n - 29]) * xy35

ElemNodeLists = np.zeros((24, 4), dtype=int)
ElemNodeLists[0] = [1, 2, 9, 8]
for e in range(2, 7):     # elements 2..6
    ElemNodeLists[e - 1] = ElemNodeLists[e - 2] + 1
ElemNodeLists[6] = ElemNodeLists[5] + 2
for e in range(8, 13):    # elements 8..12
    ElemNodeLists[e - 1] = ElemNodeLists[e - 2] + 1
ElemNodeLists[12] = ElemNodeLists[11] + 2
for e in range(14, 19):   # elements 14..18
    ElemNodeLists[e - 1] = ElemNodeLists[e - 2] + 1
ElemNodeLists[18] = ElemNodeLists[17] + 2
for e in range(20, 25):   # elements 20..24
    ElemNodeLists[e - 1] = ElemNodeLists[e - 2] + 1
 
# convert to a plain list of lists, which is what the module iterates over
ElemNodeLists = ElemNodeLists.tolist()
 

numnod = len(NodeCoordinates)          # 35
numele = len(ElemNodeLists)            # 24
 
eletyp = ["Quad4"] * numele
elemat = [[1000.0, 0.25]] * numele     
elefab = [[1.0]] * numele              
eleopt = [0]                          

K = MembraneMasterStiffness(NodeCoordinates, eletyp, ElemNodeLists,
                            elemat, elefab, eleopt)
 

eigvals = np.linalg.eigvalsh(K)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)

print("Eigenvalues of K=")
print(eigvals)
print(eigvals.shape)


Eigenvalues of K=
[   0.            0.            0.          102.3348758   114.75651474
  147.22457448  163.74275506  188.41286988  264.61514045  289.10463428
  322.71646772  354.46460176  398.43763467  461.08368605  508.48651585
  543.52720071  556.03114225  627.06922073  674.79451888  693.43326369
  729.43483653  732.80526128  799.19220538  850.41424037  865.27620543
  899.546232    917.78083006  952.35380869  983.34654337  997.2219604
 1025.03701802 1081.90287077 1122.16765215 1198.27131293 1217.54265727
 1248.34131525 1281.17430236 1332.4748084  1352.38058335 1398.20261061
 1479.72691008 1521.00613326 1580.4324351  1617.49835044 1658.39754438
 1695.46329336 1778.85283608 1854.18975242 1873.45738162 1945.8667904
 1987.68861854 2058.11377913 2099.81461061 2229.0142277  2282.87240852
 2295.75674077 2448.16633604 2655.80332784 2756.32373342 2957.15593237
 3102.63932959 3172.09176929 3321.2071933  3424.02859688 3582.17484396
 3898.85783291 4328.65994653 4478.74060947 5199.45748687 6479